# 02 · Limpieza de datos

Preparo los datos para el modelo de riesgo de lesión. Parto de los CSV originales de `data/raw/`, aplico las decisiones tomadas en la exploración (`01-exploracion-datasetNBA.ipynb`) y guardo una tabla limpia en `data/nba_load.parquet`, que es la que carga `project-model.ipynb`.

Cada paso queda registrado con el número de filas que afecta.

In [3]:
#importación de herramientas y librerías necesarias para el análisis de datos
from pathlib import Path
import numpy as np #librería de cálculo númerico sobre la que funciona pandas
import pandas as pd

RAW = Path("..")/"data"/"raw"
OUT = Path("..")/"data"/"nba_load.parquet" #es donde se guardará el fichero limpio en project/data 
pd.set_option("display.max_columns", None)

In [4]:
#Cargar el fichero de datos en bruto
player_stats = pd.read_csv(RAW/"playerStatistics.csv", low_memory=False) #low_memory=False para evitar el warning de memoria
players = pd.read_csv(RAW/"players.csv", low_memory=False)

#crear lista vacía y en cada paso se va a apuntar una línea con tres datos:
#- que se hizo, - cuantas filas afectó, - por qué se hizo. 
informe =[] #Esta es la lista vacía

print("Filas iniciales de player_stats:", len(player_stats))

Filas iniciales de player_stats: 1669922


In [6]:
#Empezamos a limpiar el fechero y a filtrar los datos que me interesan

#PRIMERO 
#Fechas: solo quiero desde la temporada 2015-16 
player_stats["gameDate"] = pd.to_datetime(player_stats["gameDate"], errors="coerce") #convierto la columna de fechas a tipo fecha, si no puede convertirlo lo pone como NaT
n = len(player_stats) #guardo el número de filas inicial
player_stats = player_stats[player_stats["gameDate"] >= "2015-10-01"] #filtro las filas que cumplen la condición

#Añado a la lista de informe lo que se ha hecho:
informe.append(["Filtrado de fechas: partidos anteriores al 01/10/2015", n-len(player_stats), "Se han eliminado los partidos anteriores a la temporada 2015-16 (fuera de mi ventana de trabajo)"])
#-----------------------------------------------------------

#SEGUNDO
#Tipo de partido: solo quiero partidos de temporada regular y  playoffs. 
#Voy a unificar los nombres de los tipos de partido y quedarme con los que me interesen. 
mapa_tipo_partido = {
    "Regular Season": "Regular Season", "Playoffs": "Playoffs", "Play-in Tournament": "Play-in",
    "NBA Emirates Cup": "NBA Cup", "Emirates NBA Cup": "NBA Cup", "NBA Cup": "NBA Cup",
    "in-season-knockout": "NBA Cup", "Preseason": "Preseason", "Pre Season": "Preseason",
    "All-Star Game": "All-Star",
}
player_stats["gameType"] = player_stats["gameType"].map(mapa_tipo_partido)
n = len(player_stats)
player_stats = player_stats[player_stats["gameType"].isin(["Regular Season", "Playoffs", "Play-in", "NBA Cup"])]

#Añado a la lista de informe lo que se ha hecho:
informe.append(("No he inlcuido Pretemporada, All-Star y otros", n - len(player_stats), "No generan carga competitiva real"))
#-----------------------------------------------------------

#TERCERO
#Jugadores sin indetificar
n=len(player_stats)
player_stats = player_stats.dropna(subset=["personId"]) #elimino las filas que no tienen personId

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Eliminación de jugadores sin personId", n - len(player_stats), "No se puede identificar al jugador, no se puede analizar"))

player_stats["personId"] = player_stats["personId"].astype(int) #convierto a int la columna personId para poder hacer el merge con players

#-----------------------------------------------------------

print("Filas finales de player_stats:", len(player_stats))
player_stats["gameType"].value_counts() #veo cuántas filas hay de cada tipo de partido

Filas finales de player_stats: 366218


gameType
Regular Season    338439
Playoffs           24959
NBA Cup             1822
Play-in              998
Name: count, dtype: int64

In [7]:
pd.DataFrame(informe, columns=["Paso", "Filas afectadas", "Decisión y motivo"])

,Paso,Filas afectadas,Decisión y motivo
0,Filtrado de fechas: partidos anteriores al 01/...,1273392,Se han eliminado los partidos anteriores a la ...
1,"No he inlcuido Pretemporada, All-Star y otros",30312,No generan carga competitiva real
2,Eliminación de jugadores sin personId,0,"No se puede identificar al jugador, no se pued..."


### CONVERTIR MINUTOS

Ahora voy a convertir ´numMinutes´ de texto a número, incluidos los valores en formato mm:ss.

La **razón** es que es mi variable de carga, mientras sea texto no la puedo sumar y la carga de 14 días es justamente una suma de minutos.

#### Lo que espero ver: 

- **Filas covertidas**: algo menos de 2.097, porque algunas eran de pretemporada o enteriores al 2015 y ya las he quitado anteriormente. 
- ´describe()´: un resumen de la columna. El maximo deberia rondar los 60 minutos ya que un partido dura eso con prórrogas aproximadamente y el mínimo, 0. Si hay valores negativos hay algo mal.  

In [9]:
#PRIMERO: convertir la columna de minutos jugados a decimal.
#Trato la columna como texto y localizo las filas con formato mm:ss
minutos_txt = player_stats["numMinutes"].astype("string")
es_mmss = minutos_txt.str.contains(":", na=False) #MARCA TRUE si la fila tiene formato mm:ss, FALSE si no lo tiene o es NaN

#SEGUNDO: convierto las filas normales a número
#Las filas normales ("22.0") se convierten directamente a número
minutos = pd.to_numeric(minutos_txt.where(~es_mmss), errors="coerce")
#~es_mmss significa "no es mm:ss", el simbolo ~ convierte el True en False y viceversa. La función where() devuelve los valores de la serie donde la condición es True y NaN donde es False. Por lo tanto, minutos_txt.where(~es_mmss) devuelve los valores de minutos_txt donde no es mm:ss y NaN donde sí lo es. Luego, pd.to_numeric() convierte esos valores a números, y los NaN se mantienen como NaN.
#luego, pd.to_numeric() convierte esos valores a números, y los NaN se mantienen como NaN.

#TERCERO: convierto las filas mm:ss a decimal
# 3) Las filas mm:ss se parten por los dos puntos: minutos + segundos/60
#--------------------------------------------------------------------------------
#HA DADO ERROR: EXPLICACIÓN ABAJO
######partes = minutos_txt[es_mmss].str.split(":", expand=True).astype(float)
######minutos[es_mmss] = partes[0] + partes[1] / 60
#--------------------------------------------------------------------------------
if es_mmss.any():
    partes = minutos_txt[es_mmss].str.split(":", expand=True).astype(float)
    minutos[es_mmss] = partes[0] + partes[1] / 60

player_stats["numMinutes"] = minutos.astype(float)

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Minutos en formato mm:ss convertidos a decimal", int(es_mmss.sum()), "Se convierten, no se eliminan"))

print("Filas convertidas de mm:ss:", es_mmss.sum())
player_stats["numMinutes"].describe()

Filas convertidas de mm:ss: 0


count    301653.000000
mean         22.240825
std          10.741454
min           0.000000
25%          14.530000
50%          23.000000
75%          31.000000
max          64.000000
Name: numMinutes, dtype: float64

**Lectura del `describe()` de los minutos:**

- **`count` 301.653 de 366.218 filas.** Las 64.565 filas restantes tienen los minutos vacíos: son jugadores que estaban convocados pero **no jugaron** (lesión, decisión del entrenador, otros motivos). No es un error de los datos. Los rellenaré con 0 más adelante, porque ese día su carga fue cero, y el motivo lo sacaré de la columna `comment`.
- **Media 22,2 y mediana 23 minutos** entre quienes jugaron. Es un valor razonable para la NBA, donde los titulares juegan unos 30-36 minutos y los suplentes bastante menos.
- **Máximo 64 minutos.** Es plausible: un partido dura 48 minutos y cada prórroga añade 5, así que 64 corresponde a un partido con tres prórrogas. No lo trato como valor atípico.
- **Mínimo 0.** Hay jugadores que constan como jugados con 0 minutos (por ejemplo, salieron solo unos segundos). Los mantengo.
- **Conclusión:** tras la conversión, `numMinutes` es numérica y sus valores están dentro de lo posible en un partido, así que la puedo usar para calcular la carga de 14 días.

ERROR ENCOTRADO Y SOLICIONADO:

**Minutos en formato mm:ss:** en la exploración vi que algunas filas guardaban los minutos como texto `mm:ss` (por ejemplo `34:12`) en lugar de como número decimal. Al aplicar la conversión, la celda dio un error (`KeyError: 0`). El motivo es que **en mi ventana (2015-16 → 2025-26) no hay ninguna fila en formato mm:ss**: todas eran de temporadas antiguas y ya las había quitado el filtro de fechas. Al no haber filas que convertir, `split(":")` devolvía una tabla sin columnas, y el código no encontraba la columna de los minutos (`partes[0]`).

**Corrección:** la conversión solo se hace si existe al menos una fila con ":" (`if es_mmss.any():`). Mantengo el paso en lugar de borrarlo, como con `personId`: si una versión futura del dataset trae minutos en mm:ss, se convertirán a decimal